# Week 8 — Evaluation Expansion

Compare Linear Regression, Decision Tree, Random Forest, and XGBoost using R²,
MAPE, MdAPE, and MAE. Check which actual price ranges have lower percentage errors.
Use the predictions already saved by Week 7, keeping the same test properties.

**Before running:** save this notebook in `notebooks/`. Your final Week 7 run must
have created `data/processed/advanced_model_predictions.csv`. Use the export from
your chosen model settings and data snapshot. If the file is missing, run the
Week 7 save-results cell after the final models have been trained. That cell is
the section titled `9. Save results and predictions` in the uploaded notebook.

This notebook needs only the prediction CSV and the libraries already used in
the project. Running it creates `metrics_summary.csv` and `price_band_metrics.csv`.
The notebook is numbered 07 because your advanced-model notebook is numbered 06.

## 1. Import libraries

Use pandas and NumPy for the calculations and Matplotlib for the comparison graph.
Keep the familiar R² function from earlier weeks.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score
from IPython.display import display, Markdown

## 2. Load the saved predictions

Each row contains one property's actual ClosePrice and the four models' estimates.
Keeping them together preserves the comparison across the same properties.
Use your final Week 7 export so the chosen XGBoost settings are represented.

In [ ]:
predictions = pd.read_csv(
    "../data/processed/advanced_model_predictions.csv",
    parse_dates=["CloseDate"]
)
model_columns = ["Linear Regression", "Decision Tree", "Random Forest", "XGBoost"]
display(predictions.head())

## 3. Sanity checks

Percentage errors need positive actual prices. Require complete, finite values
for every model so all four are evaluated on exactly the same rows. Keep large
errors and negative estimates in the evaluation rather than removing them.
Confirm that the saved rows cover one test month.

In [ ]:
numeric_columns = ["ActualClosePrice"] + model_columns
assert set(["CloseDate"] + numeric_columns).issubset(predictions.columns)
assert len(predictions) > 1, "At least two properties are needed for R²."

predictions[numeric_columns] = predictions[numeric_columns].apply(
    pd.to_numeric, errors="raise"
)
assert np.isfinite(predictions[numeric_columns].to_numpy()).all()
assert predictions["ActualClosePrice"].gt(0).all()

predictions["CloseDate"] = pd.to_datetime(predictions["CloseDate"], errors="raise")
assert predictions["CloseDate"].notna().all()
test_months = predictions["CloseDate"].dt.to_period("M")
assert test_months.nunique() == 1, "Use predictions from one test month."
test_month = test_months.iloc[0]

print("Test month:", test_month)
print("Test properties:", len(predictions))
print("Data checks passed.")

## 4. Calculate the evaluation metrics

| Metric | Meaning | Better direction |
|---|---|---|
| R² | Squared-error performance relative to predicting the test mean. | Higher |
| MAPE | Mean absolute percentage error. | Lower |
| MdAPE | Median absolute percentage error; describes typical relative error. | Lower |
| MAE | Mean absolute error in dollars. | Lower |

For example, predicting $550,000 for a $500,000 sale gives a $50,000 absolute error
and a 10% absolute percentage error. The code multiplies by 100, so a value of 10
in the percentage columns means 10%, rather than 0.10.

AVM section 08 recommends MdAPE as the headline metric because a few extreme
misses can strongly affect averages. A MdAPE of 10% means roughly half the homes
have absolute percentage errors no greater than 10%. MAPE still helps reveal
larger errors in the tail, while MAE communicates the average dollar error.

In [ ]:
actual_prices = predictions["ActualClosePrice"]
percentage_errors = pd.DataFrame(index=predictions.index)
results = []

for model in model_columns:
    estimated_prices = predictions[model]
    absolute_error = (actual_prices - estimated_prices).abs()
    percentage_errors[model] = absolute_error / actual_prices * 100

    results.append({
        "Model": model,
        "Test_R2": r2_score(actual_prices, estimated_prices),
        "MAPE_percent": percentage_errors[model].mean(),
        "MdAPE_percent": percentage_errors[model].median(),
        "MAE_dollars": absolute_error.mean(),
        "TestRows": len(predictions)
    })

metrics_summary = pd.DataFrame(results)
display(metrics_summary.round({
    "Test_R2": 4, "MAPE_percent": 2, "MdAPE_percent": 2, "MAE_dollars": 2
}))

## 5. Graph the percentage errors

Compare the mean and median percentage errors for each model. Lower bars mean
smaller errors. A much higher MAPE than MdAPE can indicate some unusually large
percentage errors; it does not by itself explain their cause.

In [ ]:
ax = metrics_summary.set_index("Model")[["MAPE_percent", "MdAPE_percent"]].plot.bar(
    figsize=(9, 5), color=["steelblue", "seagreen"]
)
ax.set_title("Week 8: Test Percentage Errors")
ax.set_ylabel("Absolute percentage error (%)")
ax.set_xlabel("")
ax.legend(["MAPE", "MdAPE"])
for bars in ax.containers:
    ax.bar_label(bars, fmt="%.2f", padding=3)
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 6. Compare actual price bands

AVM section 08 calls for checking errors across price ranges. Group properties
using actual sale prices, then compare each model within those same groups.
These fixed dollar bands are descriptive choices, not tuned parameters.

Lower boundaries are included and upper boundaries are excluded: a $500,000
property belongs in the $500k–<$1M band. Report the number of properties because
results for small groups can be less stable. Empty bands are skipped.

In [ ]:
price_labels = ["Under $500k", "$500k–<$1M", "$1M–<$2M", "$2M–<$5M", "$5M+"]
predictions["PriceBand"] = pd.cut(
    actual_prices,
    bins=[0, 500_000, 1_000_000, 2_000_000, 5_000_000, np.inf],
    labels=price_labels,
    right=False
)

band_results = []
for band in price_labels:
    in_band = predictions["PriceBand"].eq(band)
    if not in_band.any():
        continue
    for model in model_columns:
        errors = percentage_errors.loc[in_band, model]
        band_results.append({
            "PriceBand": band,
            "Model": model,
            "Properties": int(in_band.sum()),
            "MAPE_percent": errors.mean(),
            "MdAPE_percent": errors.median()
        })

price_band_metrics = pd.DataFrame(band_results)
display(price_band_metrics.round(2))

## 7. View the price-band comparison side by side

The compact table compares median percentage error for all four models. Keep
the bands in price order and show their sample counts alongside the results.
The lowest value in a row identifies the best MdAPE for that price band.

In [ ]:
band_mdape = price_band_metrics.pivot(
    index="PriceBand", columns="Model", values="MdAPE_percent"
).reindex(index=price_labels, columns=model_columns).dropna(how="all")

band_counts = predictions["PriceBand"].value_counts().reindex(band_mdape.index)
band_comparison = band_mdape.copy()
band_comparison.insert(0, "Properties", band_counts)
display(band_comparison.round(2))

## 8. Save the evaluation results

`metrics_summary.csv` is the required Week 8 CSV deliverable. The additional
price-band CSV preserves the breakdown used to write the findings. Keep the
unrounded values in the files; rounding above is for display only.

In [ ]:
metrics_summary.to_csv("../data/processed/metrics_summary.csv", index=False)
price_band_metrics.to_csv("../data/processed/price_band_metrics.csv", index=False)
print("Saved metrics_summary.csv and price_band_metrics.csv.")

## 9. Summary and findings

Use the actual results to generate a short paragraph. A model can rank best on
R² and differently on MdAPE because the metrics describe different aspects of
prediction error. The price-band findings below refer to the lowest-MdAPE model.

In [ ]:
best_mdape = metrics_summary.loc[metrics_summary["MdAPE_percent"].idxmin()]
best_r2 = metrics_summary.loc[metrics_summary["Test_R2"].idxmax()]
best_model_bands = price_band_metrics.loc[
    price_band_metrics["Model"].eq(best_mdape["Model"])
]
lowest_band = best_model_bands.loc[best_model_bands["MdAPE_percent"].idxmin()]
highest_band = best_model_bands.loc[best_model_bands["MdAPE_percent"].idxmax()]

display(Markdown(
    f"This week, I evaluated all four models on **{len(predictions):,} properties** "
    f"from **{test_month}**, adding MAPE, MdAPE, and MAE alongside R². "
    f"**{best_mdape['Model']}** had the lowest MdAPE at "
    f"**{best_mdape['MdAPE_percent']:.2f}%**, with MAPE of "
    f"**{best_mdape['MAPE_percent']:.2f}%** and MAE of "
    f"**${best_mdape['MAE_dollars']:,.0f}**. "
    f"**{best_r2['Model']}** had the highest test R² at "
    f"**{best_r2['Test_R2']:.4f}**. For the lowest-MdAPE model, median percentage "
    f"error was lowest in **{lowest_band['PriceBand']}** "
    f"(**{lowest_band['MdAPE_percent']:.2f}%**, {lowest_band['Properties']} properties) "
    f"and highest in **{highest_band['PriceBand']}** "
    f"(**{highest_band['MdAPE_percent']:.2f}%**, {highest_band['Properties']} properties). "
    f"These comparisons show how performance differs across metrics and price ranges."
))

These findings describe the saved test month and the retained properties after
Notebook 02's training-based price cutoffs. They do not cover the excluded extreme
sales. Small price-band samples deserve caution. Check additional historical
cutoffs and later months before claiming that a model or price range consistently
performs best. Geographic breakdowns and rolling backtests remain follow-up work.
Week 8 measures the existing models; use earlier validation data for later tuning.

**References:** project timeline, Week 8; AVM best-practices section 08 (MdAPE as
the headline metric, percentage and dollar errors, and price-band breakdowns),
sections 09–11 (comparison, documentation, and reproducibility); scikit-learn
[MAPE documentation](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.mean_absolute_percentage_error.html)
and [MAE documentation](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.mean_absolute_error.html).